# AMP Generative Pipeline — Executable Colab (Data → Classifiers → Generation → Pareto)

**Scope of this notebook:** everything that can genuinely run end-to-end in a Colab session — FASTA merge/dedup, CD-HIT clustering, the two RF oracle classifiers (AMP activity, hemolysis), the VAE and Transformer generators, and NSGA-II multi-objective Pareto ranking.

**Explicitly out of scope here:** ESMFold structural validation and the MARTINI3/GROMACS coarse-grained MD pipeline are **WSL2-only** (martinize2, insane, the hand-tuned `dt=0.01` production settings, and the 100ns×8-candidate runs are not things you want a Colab cell accidentally re-triggering). Those live in `amp_md_pilot_fresh.ipynb` / `run_md.py` / `analyze_md.py` etc. on your local WSL2 environment. The last cell of this notebook writes the exact file this notebook hands off to that pipeline (`pareto_v3_passing.csv`, biologically-filtered).

**Provenance flags — read before trusting this as "the" pipeline:**

1. **`amp_classifier_v5.pkl`** (the oracle actually used downstream, AUC 0.9079, Swiss-Prot hard negatives) — the training script for v5 specifically was not among the files uploaded to this project. This notebook trains **v4** (`module2_classifier_v4.py` logic: grouped 3-way split, multi-seed stability check) verbatim, since that's the last version with a script present. If you have the v5 training script, swap it in before trusting downstream candidate scores as "v5-equivalent."
2. **`vae_model_v3_fixed.pth`** (posterior-collapse-fixed VAE, referenced throughout `module6`/`module10`) — the exact "v3 fix" script wasn't uploaded either. This notebook runs `module5_vae_retrain.py`'s KL-annealed retrain (which targets the same collapse problem) and labels its output honestly as a **reconstruction**, not a guaranteed match to the checkpoint your other results were scored against.
3. **Transformer generator** — `transformer1.py` (the file you uploaded under that name) is the **e250 / Kaggle branch**, which your own `amp_project_state_summary.md` already flagged as archived (zero overlap with the current Pareto front, tied to the abandoned `FINAL_CANDIDATES.csv` branch). It is **deliberately excluded** from this notebook. Instead this notebook trains the base `module4_transformer.py` architecture and adds an explicit memorization check (train-set exact-match filter) before candidates go anywhere, matching the documented "8.7% memorized" finding — but this is again a reconstruction, not the literal `transformer_v5_generated_filtered.csv`-producing script, which wasn't uploaded.

Net effect: this notebook is internally consistent and runnable, but its classifier/VAE/Transformer outputs are **not guaranteed byte-identical** to the artifacts already scored in your final report. Treat this as a clean, from-scratch reproduction path — re-validate before swapping its outputs into anything already written up.


## 0. Environment setup

In [ ]:
!pip install -q modlamp pymoo biopython
!apt-get -qq install -y cd-hit
import os, random, re, pickle, itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

random.seed(42); np.random.seed(42); torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

BASE = "/content/amp_pipeline"
for d in ["data", "models", "results", "figures"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")


## 1. Data curation

Upload your APD6/dbAMP3/CAMPR4 FASTA/CSV sources (or the FASTA files already in this project: `pos.fa.txt`, `neg.fa.txt`, `pos1.fa.txt`, `negv.fa.txt` are the HemoPI-1 toxicity sources — the AMP-activity sources are the APD6/dbAMP3/CAMPR4 files referenced in `module12b`/`module13`, upload those here).

This cell reproduces the validated merge logic from `module12b_new_added_data.py` + `module13_merge_three_sources.py`: parse, clean (valid AA only, length 5–60), deduplicate by exact sequence, tag source, then generate matched-length random negatives (same approach used throughout — scrambled/random Swiss-Prot-style negatives, **not** the v5 oracle's actual Swiss-Prot hard-negative set, which is part of gap #1 above).

In [ ]:
from google.colab import files as colab_files

print("Upload AMP-positive FASTA source files (APD6 natural/animal/etc., dbAMP3, CAMPR4).")
print("Skip this cell if amp_dataset_v3.csv is already uploaded directly.")
uploaded = colab_files.upload()
for fname in uploaded:
    dest = f"{BASE}/data/{fname}"
    if not os.path.exists(dest):
        os.rename(fname, dest)
print("Uploaded:", list(uploaded.keys()))


In [ ]:
def parse_fasta(filepath):
    seqs = []
    with open(filepath) as f:
        cur = []
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if cur:
                    seqs.append("".join(cur)); cur = []
            elif line:
                cur.append(line)
        if cur:
            seqs.append("".join(cur))
    return seqs

def clean_sequence(seq, min_len=5, max_len=60):
    seq = seq.upper().strip()
    if not (min_len <= len(seq) <= max_len):
        return None
    if not set(seq).issubset(VALID_AA):
        return None
    return seq

records, seen = [], set()
for fname in os.listdir(f"{BASE}/data"):
    path = f"{BASE}/data/{fname}"
    if fname.endswith((".fasta", ".fa", ".fa.txt", ".txt")):
        raw = parse_fasta(path)
    elif fname.endswith(".csv"):
        raw = pd.read_csv(path).iloc[:, 0].astype(str).tolist()
    else:
        continue
    n_valid = 0
    for s in raw:
        c = clean_sequence(s)
        if c and c not in seen:
            seen.add(c)
            records.append({"sequence": c, "source": fname})
            n_valid += 1
    print(f"{fname}: {n_valid} valid unique sequences")

positives = pd.DataFrame(records)
print(f"\nTotal unique positive AMPs: {len(positives)}")


In [ ]:
# Negatives: matched-length random sequences (same construction as module13)
aa_list = list(VALID_AA)
neg_seqs = ["".join(random.choices(aa_list, k=len(s))) for s in positives["sequence"]]
negatives = pd.DataFrame({"sequence": neg_seqs, "source": "random_negative"})

positives["label"] = 1
negatives["label"] = 0
dataset_v3 = pd.concat([positives, negatives], ignore_index=True).drop_duplicates("sequence").reset_index(drop=True)
dataset_v3.to_csv(f"{BASE}/data/amp_dataset_v3.csv", index=False)
print(f"Positives: {(dataset_v3.label==1).sum()} | Negatives: {(dataset_v3.label==0).sum()} | Total: {len(dataset_v3)}")


### 1a. CD-HIT clustering (leakage-safe grouping)

Reproduces `module14a/b/c`: export positives to FASTA, cluster at 80% identity, assign `cluster_id` (CD-HIT for sequences ≥11aa, exact-match grouping for the short sequences CD-HIT silently drops — this was a real bug you caught in the original pipeline, preserved here deliberately).

In [ ]:
pos_df = dataset_v3[dataset_v3.label == 1].reset_index(drop=True)
fasta_path = f"{BASE}/data/positives_for_clustering.fasta"
with open(fasta_path, "w") as f:
    for i, row in pos_df.iterrows():
        f.write(f">seq_{i}\n{row['sequence']}\n")

!cd-hit -i {fasta_path} -o {BASE}/data/positives_clustered.fasta -c 0.8 -n 5 -d 0 -M 800 -T 4


In [ ]:
clstr_path = f"{BASE}/data/positives_clustered.fasta.clstr"
seq_to_cluster = {}
current_cluster = None
with open(clstr_path) as f:
    for line in f:
        line = line.strip()
        if line.startswith(">Cluster"):
            current_cluster = int(line.split()[-1])
        else:
            m = re.search(r">seq_(\d+)\.\.\.", line)
            if m:
                seq_to_cluster[int(m.group(1))] = current_cluster

pos_df["seq_len"] = pos_df["sequence"].str.len()
pos_df["cdhit_cluster"] = pos_df.index.map(seq_to_cluster)

max_cluster = pos_df["cdhit_cluster"].max()
max_cluster = int(max_cluster) if pd.notna(max_cluster) else -1
short_mask = pos_df["cdhit_cluster"].isna()
if short_mask.sum() > 0:
    short = pos_df[short_mask].copy()
    short["exact_group"] = pd.factorize(short["sequence"])[0]
    pos_df.loc[short_mask, "cdhit_cluster"] = (short["exact_group"] + max_cluster + 1).values
    print(f"{short_mask.sum()} short sequences (CD-HIT word-size floor) grouped by exact match "
          f"into {short['exact_group'].nunique()} clusters")

pos_df = pos_df.rename(columns={"cdhit_cluster": "cluster_id"}).drop(columns=["seq_len"])
neg_df = dataset_v3[dataset_v3.label == 0].copy()
neg_df["cluster_id"] = range(int(pos_df.cluster_id.max()) + 1,
                              int(pos_df.cluster_id.max()) + 1 + len(neg_df))

dataset_v4 = pd.concat([pos_df, neg_df], ignore_index=True)
dataset_v4.to_csv(f"{BASE}/data/amp_dataset_v4_clustered.csv", index=False)
print(f"\nUnique clusters: {dataset_v4.cluster_id.nunique()} across {len(dataset_v4)} sequences "
      f"({100*(1 - dataset_v4.cluster_id.nunique()/len(dataset_v4)):.1f}% grouped as near-duplicates)")


## 2. AMP activity classifier

**Reconstructs `module2_classifier_v4.py` (grouped 3-way split, multi-seed stability check) — see gap #1 in the header.** This is the v4 methodology, run on whatever data you fed it above. It is not guaranteed to reproduce v5's Swiss-Prot-hard-negative AUC of 0.9079.

In [ ]:
from modlamp.descriptors import GlobalDescriptor
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, roc_auc_score

def compute_features(sequences):
    desc = GlobalDescriptor(sequences)
    desc.calculate_charge(ph=7.0, amide=True); charge = desc.descriptor.copy()
    desc.calculate_MW(amide=True); mw = desc.descriptor.copy()
    desc.hydrophobic_ratio(); hydro = desc.descriptor.copy()
    desc.isoelectric_point(amide=True); pi = desc.descriptor.copy()
    desc.aromaticity(); arom = desc.descriptor.copy()
    desc.aliphatic_index(); aliph = desc.descriptor.copy()
    desc.boman_index(); boman = desc.descriptor.copy()
    return np.hstack([charge, mw, hydro, pi, arom, aliph, boman])

def grouped_3way_split(X, y, groups, test_size=0.15, val_size=0.15, random_state=42):
    s1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    trainval_idx, test_idx = next(s1.split(X, y, groups=groups))
    val_frac = val_size / (1 - test_size)
    s2 = GroupShuffleSplit(n_splits=1, test_size=val_frac, random_state=random_state)
    tv_groups, tv_y = groups[trainval_idx], y[trainval_idx]
    tr_sub, val_sub = next(s2.split(X[trainval_idx], tv_y, groups=tv_groups))
    train_idx, val_idx = trainval_idx[tr_sub], trainval_idx[val_sub]
    assert not (set(groups[train_idx]) & set(groups[val_idx])), "train/val leakage"
    assert not ((set(groups[train_idx]) | set(groups[val_idx])) & set(groups[test_idx])), "trainval/test leakage"
    return train_idx, val_idx, test_idx

df = pd.read_csv(f"{BASE}/data/amp_dataset_v4_clustered.csv")
X = compute_features(df["sequence"].tolist())
y = df["label"].values
groups = df["cluster_id"].values

seeds = [42, 7, 123, 2024, 99]
val_aucs = []
for seed in seeds:
    tr, va, te = grouped_3way_split(X, y, groups, random_state=seed)
    clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X[tr], y[tr])
    val_aucs.append(roc_auc_score(y[va], clf.predict_proba(X[va])[:, 1]))
print(f"Val AUC across seeds: mean={np.mean(val_aucs):.4f} std={np.std(val_aucs):.4f}")
if np.std(val_aucs) > 0.03:
    print("WARNING: val AUC unstable across seeds (>0.03 spread) — treat single-seed result with caution")

train_idx, val_idx, test_idx = grouped_3way_split(X, y, groups, random_state=42)
amp_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X[train_idx], y[train_idx])
test_auc = roc_auc_score(y[test_idx], amp_clf.predict_proba(X[test_idx])[:, 1])
print(f"\nFinal (seed=42) held-out test AUC: {test_auc:.4f}")
print(classification_report(y[test_idx], amp_clf.predict(X[test_idx]), target_names=["Non-AMP","AMP"]))

with open(f"{BASE}/models/amp_classifier_v4_reconstructed.pkl", "wb") as f:
    pickle.dump(amp_clf, f)


## 3. Toxicity (hemolysis) classifier

Reproduces `module_tox1_export_for_cdhit.py` + `module_tox2_rebuild_classifier.py` verbatim: pool HemoPI-1 main+validation (the four FASTA files already in this project), re-cluster (don't trust the original main/validation split — confirmed near-duplicates across it), grouped 3-way split, RF.

In [ ]:
print("Upload pos.fa.txt, neg.fa.txt, pos1.fa.txt, negv.fa.txt (HemoPI-1 main + validation)")
uploaded_tox = colab_files.upload()
for fname in uploaded_tox:
    dest = f"{BASE}/data/{fname}"
    if not os.path.exists(dest):
        os.rename(fname, dest)


In [ ]:
FILES = {"pos.fa.txt": ("main", 1), "neg.fa.txt": ("main", 0),
          "pos1.fa.txt": ("validation", 1), "negv.fa.txt": ("validation", 0)}

records = []
for fname, (source, label) in FILES.items():
    path = f"{BASE}/data/{fname}"
    for s in parse_fasta(path):
        records.append({"sequence": s, "label": label, "orig_source": source})

pooled = pd.DataFrame(records)
dupes = len(pooled) - pooled["sequence"].nunique()
cross = len(set(pooled[pooled.label==1].sequence) & set(pooled[pooled.label==0].sequence))
print(f"Pooled: {len(pooled)} | exact dupes: {dupes} (expect 0) | pos/neg cross-overlap: {cross} (expect 0)")
pooled.to_csv(f"{BASE}/data/hemopi_pooled.csv", index=False)

tox_pos = pooled[pooled.label == 1].reset_index(drop=True)
with open(f"{BASE}/data/hemopi_positives_for_cdhit.fasta", "w") as f:
    for i, seq in enumerate(tox_pos["sequence"]):
        f.write(f">seq_{i}\n{seq}\n")

!cd-hit -i {BASE}/data/hemopi_positives_for_cdhit.fasta -o {BASE}/data/hemopi_positives_clustered.fasta -c 0.8 -n 5 -d 0 -M 800 -T 4


In [ ]:
clstr_path = f"{BASE}/data/hemopi_positives_clustered.fasta.clstr"
seq_to_cluster, current_cluster = {}, None
with open(clstr_path) as f:
    for line in f:
        line = line.strip()
        if line.startswith(">Cluster"):
            current_cluster = int(line.split()[-1])
        else:
            m = re.search(r">seq_(\d+)\.\.\.", line)
            if m:
                seq_to_cluster[int(m.group(1))] = current_cluster

tox_pos["cdhit_cluster"] = tox_pos.index.map(seq_to_cluster)
max_c = tox_pos["cdhit_cluster"].max()
max_c = int(max_c) if pd.notna(max_c) else -1
missing_mask = tox_pos["cdhit_cluster"].isna()
if missing_mask.sum() > 0:
    dropped = tox_pos[missing_mask].copy()
    dropped["exact_group"] = pd.factorize(dropped["sequence"])[0]
    tox_pos.loc[missing_mask, "cdhit_cluster"] = (dropped["exact_group"] + max_c + 1).values

tox_pos = tox_pos.rename(columns={"cdhit_cluster": "cluster_id"})
tox_neg = pooled[pooled.label == 0].copy()
tox_neg["cluster_id"] = range(int(tox_pos.cluster_id.max())+1, int(tox_pos.cluster_id.max())+1+len(tox_neg))
tox_final = pd.concat([tox_pos, tox_neg], ignore_index=True)

X_tox = compute_features(tox_final["sequence"].tolist())
y_tox = tox_final["label"].values
groups_tox = tox_final["cluster_id"].values

tr, va, te = grouped_3way_split(X_tox, y_tox, groups_tox, random_state=42)
tox_clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tox[tr], y_tox[tr])
test_auc_tox = roc_auc_score(y_tox[te], tox_clf.predict_proba(X_tox[te])[:, 1])
print(f"Toxicity classifier held-out test AUC: {test_auc_tox:.4f}  (reference target: 0.993)")

with open(f"{BASE}/models/toxicity_classifier_v2.pkl", "wb") as f:
    pickle.dump(tox_clf, f)


## 4. Generative models

### 4a. VAE — reconstruction of `module5_vae_retrain.py` (KL-annealed, warm-startable). See gap #2: not guaranteed identical to `vae_model_v3_fixed.pth`.

In [ ]:
AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_LIST)}
IDX_TO_AA = {i: aa for aa, i in AA_TO_IDX.items()}
MAX_LEN, VOCAB, LATENT_DIM = 60, 20, 64

def one_hot_encode(seq, max_len=MAX_LEN):
    arr = np.zeros((max_len, VOCAB), dtype=np.float32)
    for i, aa in enumerate(seq[:max_len]):
        if aa in AA_TO_IDX:
            arr[i, AA_TO_IDX[aa]] = 1.0
    return arr.flatten()

class AMPDatasetVAE(Dataset):
    def __init__(self, sequences):
        self.data = [torch.tensor(one_hot_encode(s), dtype=torch.float32) for s in sequences]
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

class PeptideVAE(nn.Module):
    def __init__(self, input_dim=1200, h1=512, h2=256, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, h1), nn.ReLU(), nn.Linear(h1, h2), nn.ReLU())
        self.fc_mu, self.fc_logvar = nn.Linear(h2, latent_dim), nn.Linear(h2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, h2), nn.ReLU(), nn.Linear(h2, h1), nn.ReLU(), nn.Linear(h1, input_dim))
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), self.fc_logvar(h)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar); return mu + torch.randn_like(std)*std
    def decode(self, z): return self.decoder(z)
    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(recon_logits, x, mu, logvar, beta=1.0):
    recon_logits = recon_logits.view(-1, MAX_LEN, VOCAB)
    target = x.view(-1, MAX_LEN, VOCAB).argmax(dim=-1)
    recon = nn.functional.cross_entropy(recon_logits.reshape(-1, VOCAB), target.reshape(-1), reduction="mean")
    kl = -0.5*torch.mean(torch.sum(1+logvar-mu.pow(2)-logvar.exp(), dim=1))
    return recon + beta*kl, recon, kl

positives_seqs = pd.read_csv(f"{BASE}/data/amp_dataset_v4_clustered.csv")
positives_seqs = positives_seqs[positives_seqs.label==1]["sequence"].tolist()
positives_seqs = [s for s in positives_seqs if len(s) <= MAX_LEN]

loader = DataLoader(AMPDatasetVAE(positives_seqs), batch_size=64, shuffle=True)
vae = PeptideVAE().to(DEVICE)
opt = optim.Adam(vae.parameters(), lr=1e-3)

N_EPOCHS, WARMUP = 60, 20
for epoch in range(1, N_EPOCHS+1):
    vae.train()
    tot = 0.0
    beta = min(1.0, epoch/WARMUP)
    for batch in loader:
        batch = batch.to(DEVICE)
        opt.zero_grad()
        recon, mu, logvar = vae(batch)
        loss, _, _ = vae_loss(recon, batch, mu, logvar, beta=beta)
        loss.backward(); opt.step()
        tot += loss.item()
    if epoch % 10 == 0:
        print(f"VAE epoch {epoch}/{N_EPOCHS} | loss={tot/len(loader):.4f} | beta={beta:.2f}")

torch.save(vae.state_dict(), f"{BASE}/models/vae_reconstructed.pth")


In [ ]:
# Generate + diversity check (module_vae_diag_diversity.py logic — catch posterior collapse before trusting output)
def decode_tensor(t):
    idx = t.view(MAX_LEN, VOCAB).argmax(dim=-1).tolist()
    return "".join(IDX_TO_AA[i] for i in idx).rstrip("A")

vae.eval()
vae_generated = []
with torch.no_grad():
    for _ in range(2000):
        z = torch.randn(1, LATENT_DIM).to(DEVICE)
        seq = decode_tensor(vae.decode(z)[0].cpu())
        if 5 <= len(seq) <= 60 and set(seq).issubset(VALID_AA):
            vae_generated.append(seq)

from collections import Counter
counts = Counter(vae_generated)
dupe_rate = 1 - len(counts)/len(vae_generated)
print(f"VAE generated: {len(vae_generated)} | unique: {len(counts)} | duplication rate: {dupe_rate*100:.1f}%")
if dupe_rate > 0.5:
    print("WARNING: >50% duplicates — consistent with posterior collapse. Do not trust downstream "
          "oracle scores on this batch until this is fixed (KL beta schedule / architecture).")
print("Top 5 most frequent:", counts.most_common(5))

pd.DataFrame({"sequence": vae_generated}).to_csv(f"{BASE}/results/vae_generated.csv", index=False)


### 4b. Transformer — base architecture from `module4_transformer.py`. Deliberately NOT `transformer1.py` (archived e250/Kaggle branch, see header).

In [ ]:
PAD_TOKEN, START_TOKEN, END_TOKEN = 20, 21, 22
VOCAB_SIZE = 23
T_MAX_LEN, D_MODEL, N_HEADS, N_LAYERS, D_FF, DROPOUT = 62, 128, 4, 4, 256, 0.1

def encode_sequence(seq):
    tokens = [START_TOKEN] + [AA_TO_IDX[aa] for aa in seq[:60] if aa in AA_TO_IDX] + [END_TOKEN]
    tokens += [PAD_TOKEN]*(T_MAX_LEN-len(tokens))
    return tokens[:T_MAX_LEN]

def decode_tokens(tokens):
    seq = ""
    for t in tokens:
        if t == END_TOKEN: break
        if t in (START_TOKEN, PAD_TOKEN): continue
        if t in IDX_TO_AA: seq += IDX_TO_AA[t]
    return seq

class TransformerDataset(Dataset):
    def __init__(self, sequences):
        self.data = [encode_sequence(s) for s in sequences]
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        t = self.data[i]
        return torch.tensor(t[:-1]), torch.tensor(t[1:])

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=T_MAX_LEN, dropout=DROPOUT):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos*div); pe[:, 1::2] = torch.cos(pos*div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return self.dropout(x + self.pe[:, :x.size(1), :])

class PeptideTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, D_MODEL, padding_idx=PAD_TOKEN)
        self.pos_encoding = PositionalEncoding(D_MODEL)
        layer = nn.TransformerDecoderLayer(D_MODEL, N_HEADS, D_FF, DROPOUT, batch_first=True)
        self.transformer = nn.TransformerDecoder(layer, N_LAYERS)
        self.fc_out = nn.Linear(D_MODEL, VOCAB_SIZE)
        self.register_buffer("memory", torch.zeros(1, 1, D_MODEL))
    def forward(self, x):
        mask = nn.Transformer.generate_square_subsequent_mask(x.size(1), device=x.device)
        pad_mask = (x == PAD_TOKEN)
        emb = self.pos_encoding(self.embedding(x))
        mem = self.memory.expand(x.size(0), -1, -1)
        out = self.transformer(emb, mem, tgt_mask=mask, tgt_key_padding_mask=pad_mask)
        return self.fc_out(out)

loader_t = DataLoader(TransformerDataset(positives_seqs), batch_size=64, shuffle=True)
transformer = PeptideTransformer().to(DEVICE)
opt_t = optim.Adam(transformer.parameters(), lr=1e-3)
crit_t = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)

for epoch in range(1, 41):
    transformer.train()
    tot = 0.0
    for x, y in loader_t:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt_t.zero_grad()
        logits = transformer(x)
        loss = crit_t(logits.view(-1, VOCAB_SIZE), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(transformer.parameters(), 1.0)
        opt_t.step()
        tot += loss.item()
    if epoch % 10 == 0:
        print(f"Transformer epoch {epoch}/40 | loss={tot/len(loader_t):.4f}")

torch.save(transformer.state_dict(), f"{BASE}/models/transformer_reconstructed.pth")


In [ ]:
# Generate + MEMORIZATION CHECK against training set (the ~8.7% exact-copy finding from the real pipeline)
transformer.eval()
transformer_generated = []
with torch.no_grad():
    for _ in range(1000):
        tokens = [START_TOKEN]
        for _ in range(60):
            x = torch.tensor([tokens], device=DEVICE)
            logits = transformer(x)[0, -1, :] / 0.8
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            if nxt == END_TOKEN: break
            tokens.append(nxt)
        seq = decode_tokens(tokens)
        if len(seq) >= 5 and set(seq).issubset(VALID_AA):
            transformer_generated.append(seq)

transformer_generated = list(dict.fromkeys(transformer_generated))  # dedupe, preserve order
training_set = set(positives_seqs)
memorized = [s for s in transformer_generated if s in training_set]
mem_rate = len(memorized) / len(transformer_generated) if transformer_generated else 0
print(f"Generated: {len(transformer_generated)} unique | exact training-set copies: {len(memorized)} ({mem_rate*100:.1f}%)")

transformer_filtered = [s for s in transformer_generated if s not in training_set]
pd.DataFrame({"sequence": transformer_filtered, "source": "Transformer_reconstructed"}).to_csv(
    f"{BASE}/results/transformer_generated_filtered.csv", index=False)
print(f"Saved {len(transformer_filtered)} memorization-filtered candidates")


## 5. Score all candidates + degenerate/low-complexity flag

Reproduces `module6_score_candidates.py`'s `flag_degenerate` (poly-runs ≥5, >60% single-residue composition, extreme charge >+10) — this is the exact check that would have caught `KLKLKLRRK`-style artifacts, so it's run here before anything reaches the Pareto step, not after.

In [ ]:
def flag_degenerate(seq):
    flags = []
    max_run = max(sum(1 for _ in g) for _, g in itertools.groupby(seq))
    if max_run >= 5:
        flags.append(f"poly_run_{max_run}")
    for aa in set(seq):
        frac = seq.count(aa)/len(seq)
        if frac > 0.60:
            flags.append(f"high_{aa}_{frac:.0%}")
    charge = seq.count('K') + seq.count('R') - seq.count('D') - seq.count('E')
    if charge > 10:
        flags.append(f"extreme_charge_{charge:+d}")
    return ", ".join(flags) if flags else "ok"

all_candidates = list(dict.fromkeys(vae_generated + transformer_filtered))
X_cand = compute_features(all_candidates)
amp_scores = amp_clf.predict_proba(X_cand)[:, 1]
tox_scores = tox_clf.predict_proba(X_cand)[:, 1]

scored = pd.DataFrame({
    "sequence": all_candidates,
    "amp_score": amp_scores,
    "tox_score": tox_scores,
    "charge": [s.count('K')+s.count('R')-s.count('D')-s.count('E') for s in all_candidates],
    "hydro_pct": [sum(1 for aa in s if aa in 'LVAIFMW')/len(s)*100 for s in all_candidates],
    "flags": [flag_degenerate(s) for s in all_candidates],
    "source": ["VAE" if s in set(vae_generated) else "Transformer" for s in all_candidates],
})
n_flagged = (scored.flags != "ok").sum()
print(f"Scored {len(scored)} candidates | flagged as degenerate/low-complexity: {n_flagged}")
scored.to_csv(f"{BASE}/results/all_candidates_scored.csv", index=False)


## 6. Multi-objective Pareto ranking (NSGA-II non-dominated sorting)

Direct port of `module10_multiobjective_v3.py`'s objective functions — AMP activity (maximize), Chou-Fasman helix propensity + amphipathic bonus (maximize), hemolysis risk (minimize). Degenerate-flagged sequences are excluded before ranking, which the original v3 script did not do at this stage — this closes that gap.

In [ ]:
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

def compute_helix_propensity(seq):
    cf = {'A':1.45,'R':0.98,'N':0.73,'D':0.98,'C':0.77,'Q':1.17,'E':1.53,'G':0.53,'H':1.24,
          'I':1.00,'L':1.34,'K':1.07,'M':1.20,'F':1.12,'P':0.59,'S':0.79,'T':0.82,'W':1.14,'Y':0.61,'V':1.14}
    if len(seq) < 5: return 0.0
    raw = np.mean([cf.get(aa, 1.0) for aa in seq])
    norm = (raw - 0.53) / (1.53 - 0.53)
    cationic, hydrophob = set('KR'), set('LVAIFM')
    amph = sum(1 for i in range(len(seq)-3)
               if (seq[i] in cationic and seq[i+3] in hydrophob) or (seq[i] in hydrophob and seq[i+3] in cationic))
    bonus = min(amph / max(len(seq), 1), 0.3)
    return float(np.clip(norm + bonus, 0.0, 1.0))

clean = scored[scored.flags == "ok"].reset_index(drop=True)
clean["helix_score"] = clean["sequence"].apply(compute_helix_propensity)
clean["hemo_risk"] = clean["tox_score"]
clean["safety_score"] = 1 - clean["hemo_risk"]

F = np.column_stack([-clean["amp_score"].values, -clean["helix_score"].values, clean["hemo_risk"].values])
fronts = NonDominatedSorting().do(F)
pareto_idx = fronts[0]
clean["pareto_rank"] = np.nan
for rank, front in enumerate(fronts):
    clean.loc[front, "pareto_rank"] = rank + 1

print(f"Pareto front: {len(pareto_idx)} sequences out of {len(clean)} clean candidates")
pareto_df = clean.iloc[pareto_idx].sort_values("amp_score", ascending=False).reset_index(drop=True)
pareto_df.to_csv(f"{BASE}/results/pareto_candidates.csv", index=False)
clean.to_csv(f"{BASE}/results/all_candidates_scored_ranked.csv", index=False)

fig, ax = plt.subplots(figsize=(8,6))
sc = ax.scatter(clean.amp_score, clean.safety_score, c=clean.helix_score, cmap="RdYlGn", alpha=0.4, s=15)
ax.scatter(pareto_df.amp_score, pareto_df.safety_score, s=80, facecolors="none", edgecolors="black", linewidth=1.5, label="Pareto front")
plt.colorbar(sc, label="Helix propensity")
ax.set_xlabel("AMP score (activity)"); ax.set_ylabel("Safety (1 - hemolysis risk)")
ax.set_title("Activity vs Safety — Pareto front"); ax.legend()
plt.tight_layout(); plt.savefig(f"{BASE}/figures/pareto_front.png", dpi=150)
plt.show()


## 7. Biological plausibility filter → WSL2 MD handoff

Reproduces `filter_for_esmfold.py`: charge +2 to +8, hydrophobicity 30–55%. This is the file that actually leaves Colab — download it and hand it to the WSL2 MD pipeline (`amp_md_pilot_fresh.ipynb`) for ESMFold + MARTINI3/GROMACS. **Do not attempt to run ESMFold or MD in this notebook.**

In [ ]:
passing = pareto_df[
    (pareto_df.charge >= 2) & (pareto_df.charge <= 8) &
    (pareto_df.hydro_pct >= 30) & (pareto_df.hydro_pct <= 55)
].sort_values("amp_score", ascending=False).reset_index(drop=True)

print(f"Pareto front: {len(pareto_df)} -> after charge/hydrophobicity filter: {len(passing)} candidates")
print(passing[["sequence","amp_score","helix_score","safety_score","charge","hydro_pct"]].to_string(index=False))

passing.to_csv(f"{BASE}/results/pareto_v3_passing.csv", index=False)
colab_files.download(f"{BASE}/results/pareto_v3_passing.csv")
print("\nDownloaded pareto_v3_passing.csv — hand this to the WSL2 MD pipeline.")
print("Next step (outside this notebook, WSL2 only): amp_md_pilot_fresh.ipynb — ESMFold fold + verify,")
print("then martinize2 -> insane.py bilayer build -> minim/equil/production (GROMACS 2025.4, MARTINI3).")
